In [0]:
%sql
SELECT COUNT(*) FROM databricks_300941.default.dim_date;

SELECT COUNT(*) FROM databricks_300941.default.dim_rider;

SELECT COUNT(*) FROM databricks_300941.default.dim_station;

SELECT COUNT(*) FROM databricks_300941.default.fact_trip;

SELECT COUNT(*) FROM databricks_300941.default.fact_payment;

count(1)
1946607


In [0]:
%sql
SHOW TABLES IN databricks_300941.default;

database,tableName,isTemporary
default,bronze_payments,false
default,bronze_riders,false
default,bronze_stations,false
default,bronze_trips,false
default,dim_date,false
default,dim_rider,false
default,dim_station,false
default,fact_payment,false
default,fact_trip,false
,_sqldf,true


In [0]:
%sql
SELECT *
FROM databricks_300941.default.fact_payment
LIMIT 10;

payment_id,rider_id,date_key,amount
512466,20834,20201101,9.0
512467,20834,20201201,9.0
512468,20834,20210101,9.0
512469,20834,20210201,9.0
512470,20834,20210301,9.0
512471,20834,20210401,9.0
512472,20834,20210501,9.0
512473,20834,20210601,9.0
512474,20834,20210701,9.0
512475,20834,20210801,9.0


In [0]:
fact_payment.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("databricks_300941.default.fact_payment")

In [0]:
display(fact_payment)

payment_id,rider_id,date_key,amount
512466,20834,20201101,9.0
512467,20834,20201201,9.0
512468,20834,20210101,9.0
512469,20834,20210201,9.0
512470,20834,20210301,9.0
512471,20834,20210401,9.0
512472,20834,20210501,9.0
512473,20834,20210601,9.0
512474,20834,20210701,9.0
512475,20834,20210801,9.0


In [0]:
fact_payment = fact_payment.select(
    col("p.payment_id").alias("payment_id"),
    col("p.rider_id").alias("rider_id"),
    col("d.date_key").alias("date_key"),
    col("p.amount").alias("amount")
)

In [0]:
fact_payment = (
    bronze_payments.alias("p")
    .join(
        dim_date.alias("d"),
        col("p.date") == col("d.date"),
        "left"
    )
)

In [0]:
from pyspark.sql.functions import *

bronze_payments = spark.table("databricks_300941.default.bronze_payments")
dim_date = spark.table("databricks_300941.default.dim_date")